## Rag from Scratch 

### Environment 

In [ ]:
! pip install langchain langchain_community langchain_google_genai langchain-text-splitters tiktoken chromadb bs4 

In [ ]:
import os 
os.environ['LANGCHAIN_TRACING_V2'] = 'false'
os.environ['LANGCHAIN_ENDPOINT'] = 'https://api.smith.langchain.com'
os.environ['LANGCHAIN_API_KEY'] = "<your_api_key>"
os.environ['GOOGLE_API_KEY'] = "<your_api_key>"

### Packages 

In [ ]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_community.document_loaders import WebBaseLoader
from langchain_community.vectorstores import Chroma
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_text_splitters import RecursiveCharacterTextSplitter
import bs4

USER_AGENT environment variable not set, consider setting it to identify your requests.


### Indexing 

In [ ]:
# load data
loader = WebBaseLoader(
    web_paths=("https://lilianweng.github.io/posts/2023-06-23-agent/",),
    bs_kwargs=dict(
        parse_only=bs4.SoupStrainer(
            class_=("post-content", "post-title", "post-header")
        )
    ),
)
blog_docs = loader.load()

In [4]:
# splitting 
text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    chunk_size=300, 
    chunk_overlap=50)

splits = text_splitter.split_documents(blog_docs)

### Retrieval

In [5]:
# define vectorstore and then retriever 
embd = GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-001")

vectorstore = Chroma.from_documents(documents=splits, 
                                    embedding=embd)

# transforming vectorstore in retriever (number of output splitts : k = 1)
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

In [6]:
# get retrieval chuncks 
retrieval_chuncks = retriever.invoke("What is Task Decomposition?")

### Generation 

In [7]:
# Prompt
prompt = ChatPromptTemplate.from_template("""
You are an assistant for question-answering tasks.

Use the following retrieved context to answer the question.

If the answer is not in the context, say you don't know.

Context:
{context}

Question:
{question}

Answer:
""")

In [8]:
# LLM and chain 
llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0)
chain = prompt | llm

In [9]:
# Run
chain.invoke({"context": retrieval_chuncks,"question":"What is Task Decomposition?"})

AIMessage(content='Task decomposition is a process that transforms big, complex tasks into multiple smaller, simpler, and more manageable steps. This can be achieved by instructing a model to "think step by step" (Chain of Thought) or by exploring multiple reasoning possibilities at each step, creating a tree structure (Tree of Thoughts).\n\nTask decomposition can be done in several ways:\n1.  By an LLM using simple prompts like "Steps for XYZ." or "What are the subgoals for achieving XYZ?".\n2.  By using task-specific instructions, such as "Write a story outline." for writing a novel.\n3.  With human inputs.', additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019e145c-71e4-7060-9c9d-cd5f83e0d31d-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 1024, 'output_tokens': 180, 'total_tokens': 1204, 'input_token_details': {'cache_read': 0}, 'output_t